In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("New").getOrCreate()

In [0]:
employee_data = [
    (101, "Alice", 25, "HR", 45000, "Chennai", "2022-01-15", 201),
    (102, "Bob", 30, "IT", 70000, "Bangalore", "2021-06-20", 202),
    (103, "Charlie", None, "IT", None, "Chennai", "2023-03-12", 202),
    (104, "David", 28, "Finance", 65000, "Mumbai", "2020-09-18", 203),
    (105, "Eva", 35, "HR", 80000, None, "2019-05-25", 201),
    (106, "Frank", 29, "Marketing", 55000, "Hyderabad", None, 204),
    (107, "Grace", 31, "Finance", None, "Pune", "2022-12-01", 203),
    (108, "Henry", 26, "IT", 60000, "Bangalore", "2024-01-10", None)
]

employee_columns = [
    "emp_id",
    "name",
    "age",
    "department",
    "salary",
    "city",
    "joining_date",
    "manager_id"
]

employee_df = spark.createDataFrame(employee_data, employee_columns)



display(employee_df)


emp_id,name,age,department,salary,city,joining_date,manager_id
101,Alice,25,HR,45000,Chennai,2022-01-15,201
102,Bob,30,IT,70000,Bangalore,2021-06-20,202
103,Charlie,null,IT,null,Chennai,2023-03-12,202
104,David,28,Finance,65000,Mumbai,2020-09-18,203
105,Eva,35,HR,80000,null,2019-05-25,201
106,Frank,29,Marketing,55000,Hyderabad,null,204
107,Grace,31,Finance,null,Pune,2022-12-01,203
108,Henry,26,IT,60000,Bangalore,2024-01-10,null


In [0]:
department_data = [
    ("HR", "Chennai"),
    ("IT", "Bangalore"),
    ("Finance", "Mumbai"),
    ("Marketing", "Hyderabad"),
    ("Sales", "Delhi")
]

department_columns = [
    "dept_name",
    "location"
]

department_df = spark.createDataFrame(department_data, department_columns)

display(department_df)


dept_name,location
HR,Chennai
IT,Bangalore
Finance,Mumbai
Marketing,Hyderabad
Sales,Delhi


In [0]:
manager_data = [
    (201, "Robert"),
    (202, "Jennifer"),
    (203, "Michael"),
    (204, "Sophia")
]

manager_columns = [
    "manager_id",
    "manager_name"
]

manager_df = spark.createDataFrame(manager_data, manager_columns)


display(manager_df)



manager_id,manager_name
201,Robert
202,Jennifer
203,Michael
204,Sophia


- Part B – Select Operations
- 6.	Select only emp_id, name, and salary.
- 7.	Select all employees from the IT department.
- 8.	Select employee names and alias the column as Employee_Name.
- 9.	Select salary after increasing it by 10%.
- 10.	Select employees whose city is Chennai or Bangalore

In [0]:
employee_df.select("emp_id","name","salary").show()
employee_df.filter(col("department") == "IT").show()
employee_df.select(col("name").alias("Employee_Name")).show()
employee_df.select(col("salary"),(col("salary")+(0.1*col("salary"))).alias("New_Salary")).show()
employee_df.filter((col("city") == "Chennai") | (col("city") == "Bangalore")).show()



+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|   101|  Alice| 45000|
|   102|    Bob| 70000|
|   103|Charlie|  NULL|
|   104|  David| 65000|
|   105|    Eva| 80000|
|   106|  Frank| 55000|
|   107|  Grace|  NULL|
|   108|  Henry| 60000|
+------+-------+------+

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+

+-------------+
|Employee_Name|
+-------------+
|        Alice|
|          Bob|
|      Charlie|
|        David|
|          Eva|
|        Frank|
|        Grace|
|        Henry|
+-------------+


- - Part C – withColumn()
- 11.	Create a new column called bonus equal to 20% of salary.
- 12.	Create a column called annual_salary.
- 13.	Create a column called experience_bonus where:
- •	Salary > 70000 → 10000
- •	Salary between 50000 and 70000 → 5000
- •	Otherwise → 2000
- 14.	Convert employee names to uppercase.
- 15.	Add a column called country with value "India".

In [0]:
employee_df.withColumn("bonus",col("salary")+(0.2*col("salary"))).show()
employee_df.withColumn("annual_Salary",(col("salary")*12)).show()
employee_df.withColumn(
    "experience_bonus",
    when(col("salary")>70000,col("salary")+10000)
    .when(col("salary").between(50000,70000),col("salary")+5000)
    .otherwise(col("salary")+2000)
    ).show()
employee_df.withColumn("name",upper(col("name"))).show()
employee_df.withColumn("country",lit("India")).show()


+------+-------+----+----------+------+---------+------------+----------+-------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|  bonus|
+------+-------+----+----------+------+---------+------------+----------+-------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|54000.0|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|84000.0|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|   NULL|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|78000.0|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|96000.0|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|66000.0|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|   NULL|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|72000.0|
+------+-------+----+----------+------+---------+------------+----------+-------+

+------+-------

- Part D – withColumnRenamed()
- 16.	Rename department to dept.
- 17.	Rename salary to monthly_salary.
- 18.	Rename multiple columns one by one.

In [0]:
employee_df.withColumnRenamed("department","dept").show()
employee_df.withColumnRenamed("salary","monthly_salary").show()
employee_df = employee_df \
    .withColumnRenamed("emp_id","employee_id") \
    .withColumnRenamed("name","employee_name") \
    .withColumnRenamed("age","employee_age") \
    .withColumnRenamed("department","employee_department") \
    .withColumnRenamed("monthly_salary","salary") \
    .withColumnRenamed("city","employee_city") \
    .withColumnRenamed("joining_date","employee_joining_date") \
    .withColumnRenamed("manager_id","managers_id").show()



+------+-------+----+---------+------+---------+------------+----------+
|emp_id|   name| age|     dept|salary|     city|joining_date|manager_id|
+------+-------+----+---------+------+---------+------------+----------+
|   101|  Alice|  25|       HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|       IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|       IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|  Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|       HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29|Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|  Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|       IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+---------+------+---------+------------+----------+

+------+-------+----+----------+--------------+---------+------------+----------+
|emp_id|   name| age|department|monthly_s

- Part E – Filter
- 19.	Find employees earning more than 60000.
- 20.	Find employees older than 30.
- 21.	Find HR employees in Chennai.
- 22.	Find employees with NULL salary.
- 23.	Find employees whose salary is between 50000 and 70000.
- 24.	Find employees whose names start with "A".
- 25.	Find employees whose city is not NULL.

In [0]:
employee_df.filter(col("salary")>60000).show()
employee_df.filter(col("age")>30).show()
employee_df.filter(
    (col("department") == "HR") &
    (col("city") == "Chennai")
).show()
employee_df.filter(col("salary").isNull()).show()
employee_df.filter(col("salary").between(50000,70000)).show()
employee_df.filter(col("name").like("A%")).show()
employee_df.filter(col("city").isNull()).show()


+------+-----+---+----------+------+---------+------------+----------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|
+------+-----+---+----------+------+---------+------------+----------+
|   102|  Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   104|David| 28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|  Eva| 35|        HR| 80000|     NULL|  2019-05-25|       201|
+------+-----+---+----------+------+---------+------------+----------+

+------+-----+---+----------+------+----+------------+----------+
|emp_id| name|age|department|salary|city|joining_date|manager_id|
+------+-----+---+----------+------+----+------------+----------+
|   105|  Eva| 35|        HR| 80000|NULL|  2019-05-25|       201|
|   107|Grace| 31|   Finance|  NULL|Pune|  2022-12-01|       203|
+------+-----+---+----------+------+----+------------+----------+

+------+-----+---+----------+------+-------+------------+----------+
|emp_id| name|age|department|salary|

- Part F – Sort
- 26.	Sort employees by salary ascending.
- 27.	Sort employees by salary descending.
- 28.	Sort employees by department and salary.
- 29.	Sort employees by age descending.
- 30.	Display top 3 highest-paid employees.

In [0]:
employee_df.sort(col("salary").asc()).show()
employee_df.sort(col("salary").desc()).show()
employee_df.orderBy(col("department"),col("salary").desc()).show()
employee_df.sort(col("age").desc()).show()
employee_df.orderBy(col("salary").desc()).show(3)


+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
+------+-------+----+----------+------+---------+------------+----------+

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salar

- Part G – FillNA and DropNA
- 31.	Replace NULL salary with 30000.
- 32.	Replace NULL city with "Unknown".
- 33.	Replace NULL age with average age.
- 34.	Replace multiple NULL columns in one statement.
- 35.	Drop rows containing any NULL values.
- 36.	Drop rows where salary is NULL.
- 37.	Drop rows only if all values are NULL.

In [0]:
employee_df.na.fill({"salary":30000}).show()
employee_df.fillna({"city":"Unknown"}).show()
avg_age = employee_df.select(avg("age")).first()[0]
employee_df.na.fill({"age": avg_age}).show()
employee_df.fillna(0).fillna("Unknown").show()
employee_df.dropna().show()
employee_df.dropna(subset=["salary"]).show()
employee_df.dropna(thresh=1).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT| 30000|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance| 30000|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
+------+-------+----+----------+------+---------+------------+----------+

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salar

- Part H – Date Functions
- 38.	Convert joining_date to DateType.
- 39.	Extract joining year.
- 40.	Extract joining month.
- 41.	Extract joining day.
- 42.	Find employees who joined after 2022.
- 43.	Calculate years worked.
- 44.	Find employees who joined this year.
- 45.	Display current date.
- 46.	Display current timestamp.
- 47.	Add 30 days to joining date.
- 48.	Find the difference between joining date and today's date.

In [0]:
employee_df = employee_df.withColumn("joining_date",to_date(col("joining_date"), "yyyy-MM-dd"))
display(employee_df)
employee_df.withColumn("joining_year",year(col("joining_date"))).show()
employee_df.withColumn("joining_month",month(col("joining_date"))).show()
employee_df.withColumn("joining_day",dayofmonth(col("joining_date"))).show()
employee_df.filter(year(col("joining_date"))>2022).show()
employee_df.withColumn("years_worked",floor(months_between(current_date(),col("joining_date"))/12)).show()
employee_df.filter(year(col("joining_date"))== year(current_date())).show()
employee_df.select(current_date().alias("Current_date")).show(1)
spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")
employee_df.select(current_timestamp().alias("Current_time")).show(1)
employee_df.withColumn("joining_date_plus30",date_add(col("joining_date"),30)).show()
employee_df.withColumn("diff_btwn_joining_current",datediff(current_date(),col("joining_date"))).show()


emp_id,name,age,department,salary,city,joining_date,manager_id
101,Alice,25,HR,45000,Chennai,2022-01-15,201
102,Bob,30,IT,70000,Bangalore,2021-06-20,202
103,Charlie,null,IT,null,Chennai,2023-03-12,202
104,David,28,Finance,65000,Mumbai,2020-09-18,203
105,Eva,35,HR,80000,null,2019-05-25,201
106,Frank,29,Marketing,55000,Hyderabad,null,204
107,Grace,31,Finance,null,Pune,2022-12-01,203
108,Henry,26,IT,60000,Bangalore,2024-01-10,null


+------+-------+----+----------+------+---------+------------+----------+------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|joining_year|
+------+-------+----+----------+------+---------+------------+----------+------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|        2022|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|        2021|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|        2023|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|        2020|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|        2019|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|        NULL|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|        2022|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|        2024|
+------+-------+----+----------+------+----

- Part I – Joins
- 49.	Perform an inner join between Employee and Department.
- 50.	Perform a left join.
- 51.	Perform a right join.
- 52.	Perform a full outer join.
- 53.	Join Employee with Manager DataFrame.
- 54.	Display employee name along with manager name.
- 55.	Find employees whose department location matches their city.
- 56.	Find departments having no employees.

In [0]:
employee_df.join(department_df,col("department")==col("dept_name"),how="inner").show()
employee_df.join(department_df,col("department")==col("dept_name"),"left").show()
employee_df.join(department_df,col("department")==col("dept_name"),"right").show()
employee_df.join(department_df,col("department")==col("dept_name"),"full_outer").show()
employee_df.join(manager_df,"manager_id","left").show()
employee_df.join(manager_df, "manager_id", "left").select("name", "manager_name").show()
employee_df.join(department_df,col("department")==col("dept_name"),"left").filter(col("city")==col("location")).show()
department_df.join(employee_df,col("department")==col("dept_name"),"left_anti").show()

+------+-------+----+----------+------+---------+------------+----------+---------+---------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|dept_name| location|
+------+-------+----+----------+------+---------+------------+----------+---------+---------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|       HR|  Chennai|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|       IT|Bangalore|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|       IT|Bangalore|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|  Finance|   Mumbai|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|       HR|  Chennai|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|Marketing|Hyderabad|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|  Finance|   Mumbai|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-1

- Part J – Union
- Create another Employee DataFrame:
- emp_id	name	age	department	salary	city	joining_date	manager_id
- 109	Irene	27	HR	52000	Chennai	2024-03-11	201
- 110	Jack	34	Sales	72000	Delhi	2021-11-20	205
- 57.	Union the two DataFrames.
- 58.	Count total employees after union.
- 59.	Remove duplicate employees after union.
- 60.	Union by column names.

In [0]:
new_employee_data = [
    (109, "Irene", 27, "HR", 52000, "Chennai", "2024-03-11", 201),
    (110, "Jack", 34, "Sales", 72000, "Delhi", "2021-11-20", 205)
]

new_employee_df = spark.createDataFrame(
    new_employee_data,
    employee_columns
)
display(new_employee_df)


emp_id,name,age,department,salary,city,joining_date,manager_id
109,Irene,27,HR,52000,Chennai,2024-03-11,201
110,Jack,34,Sales,72000,Delhi,2021-11-20,205


In [0]:
employee_df.union(new_employee_df).show()
employee_df.union(new_employee_df).count()
employee_df.union(new_employee_df).distinct().show()
employee_df.unionByName(new_employee_df).show()

+------+-------+----+----------+------+---------+------------+----------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|
+------+-------+----+----------+------+---------+------------+----------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|
|   109|  Irene|  27|        HR| 52000|  Chennai|  2024-03-11|       201|
|   110|   Jack|  34|     Sales| 72000|    Delhi|  2021-11-20|       205|
+------+-------+----+----------+------

- Part K – Other Transformations
- 61.	Display distinct departments.
- 62.	Count employees in each department.
- 63.	Calculate average salary by department.
- 64.	Find maximum salary by department.
- 65.	Find minimum salary by department.
- 66.	Find total salary paid in each department.
- 67.	Count employees in each city.
- 68.	Display unique cities.
- 69.	Remove duplicate records.
- 70.	Drop the manager_id column.

In [0]:
employee_df.select("department").distinct().show()
employee_df.groupby(col("department")).count().show()
employee_df.groupby("department").avg("salary").show()
employee_df.groupBy("department").max("salary").show()
employee_df.groupBy("department").min("salary").show()
employee_df.groupBy("department").sum("salary").show()
employee_df.groupBy("city").count().show()
employee_df.select("city").distinct().show()
employee_df.dropDuplicates().show()
employee_df.drop(col("manager_id")).show()

+----------+
|department|
+----------+
|        HR|
|        IT|
|   Finance|
| Marketing|
+----------+

+----------+-----+
|department|count|
+----------+-----+
|        HR|    2|
|        IT|    3|
|   Finance|    2|
| Marketing|    1|
+----------+-----+

+----------+-----------+
|department|avg(salary)|
+----------+-----------+
|        HR|    62500.0|
|        IT|    65000.0|
|   Finance|    65000.0|
| Marketing|    55000.0|
+----------+-----------+

+----------+-----------+
|department|max(salary)|
+----------+-----------+
|        HR|      80000|
|        IT|      70000|
|   Finance|      65000|
| Marketing|      55000|
+----------+-----------+

+----------+-----------+
|department|min(salary)|
+----------+-----------+
|        HR|      45000|
|        IT|      60000|
|   Finance|      65000|
| Marketing|      55000|
+----------+-----------+

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|        HR|     125000|
|        IT|     130000|
|   Finance|  

- Part L – String Functions
- 71.	Convert employee names to lowercase.
- 72.	Find the length of each employee name.
- 73.	Extract the first three letters of employee names.
- 74.	Concatenate employee name and city.
- 75.	Replace all occurrences of "a" with "@" in names.

In [0]:
employee_df.select(lower(col("name"))).show()
employee_df.select(length(col("name"))).show()
employee_df.select(col("name").substr(1,1).alias("first_letter")).show()
employee_df.withColumn("name_city",concat(col("name"),col("city"))).show()
employee_df.withColumn("name",regexp_replace("name","a","@")).show()


+-----------+
|lower(name)|
+-----------+
|      alice|
|        bob|
|    charlie|
|      david|
|        eva|
|      frank|
|      grace|
|      henry|
+-----------+

+------------+
|length(name)|
+------------+
|           5|
|           3|
|           7|
|           5|
|           3|
|           5|
|           5|
|           5|
+------------+

+------------+
|first_letter|
+------------+
|           A|
|           B|
|           C|
|           D|
|           E|
|           F|
|           G|
|           H|
+------------+

+------+-------+----+----------+------+---------+------------+----------+--------------+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|     name_city|
+------+-------+----+----------+------+---------+------------+----------+--------------+
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|  AliceChennai|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|  BobBangalore|
|   103|Charlie|NULL|   

- Part M – Aggregations
- 76.	Find total salary.
- 77.	Find average salary.
- 78.	Find highest-paid employee.
- 79.	Find lowest-paid employee.
- 80.	Count employees with salary greater than 50000.

In [0]:
employee_df.select(sum("salary")).show()
employee_df.select(avg("salary")).show()
employee_df.orderBy(col("salary").desc()).limit(1).show()
min_salary = employee_df.select(min("salary")).first()[0]
employee_df.filter(col("salary") == min_salary).show()
employee_df.filter(col("salary")>50000).show()

+-----------+
|sum(salary)|
+-----------+
|     375000|
+-----------+

+-----------+
|avg(salary)|
+-----------+
|    62500.0|
+-----------+

+------+----+---+----------+------+----+------------+----------+
|emp_id|name|age|department|salary|city|joining_date|manager_id|
+------+----+---+----------+------+----+------------+----------+
|   105| Eva| 35|        HR| 80000|NULL|  2019-05-25|       201|
+------+----+---+----------+------+----+------------+----------+

+------+-----+---+----------+------+-------+------------+----------+
|emp_id| name|age|department|salary|   city|joining_date|manager_id|
+------+-----+---+----------+------+-------+------------+----------+
|   101|Alice| 25|        HR| 45000|Chennai|  2022-01-15|       201|
+------+-----+---+----------+------+-------+------------+----------+

+------+-----+---+----------+------+---------+------------+----------+
|emp_id| name|age|department|salary|     city|joining_date|manager_id|
+------+-----+---+----------+------+--------

- Part N – Advanced Questions
- 81.	Display employees whose salary is above the department average.
- 82.	Find the second highest salary.
- 83.	Find duplicate cities.
- 84.	Display employees who joined in the last two years.
- 85.	Replace NULL manager IDs with 999.
- 86.	Create a salary grade:
- •	A → Salary ≥ 80000
- •	B → Salary ≥ 60000
- •	C → Salary ≥ 40000
- •	D → Otherwise
- 87.	Create an employee ID string like EMP101.
- 88.	Find employees whose names end with "e".
- 89.	Display employees sorted by joining date.
- 90.	Find employees who have worked for more than three years.

In [0]:
dept_avg = employee_df.groupBy("department").agg(avg("salary").alias("avg_sal"))
employee_df.join(dept_avg,"department").filter(col("salary")>col("avg_sal")).show()
high_sal = employee_df.select(max("salary")).first()[0]
employee_df.filter(col("salary")< high_sal).orderBy(col("salary").desc()).limit(1).show()
ans = employee_df.groupBy("city").count()
ans.filter(col("count")>1).show()
employee_df.withColumn("exp",floor(months_between(current_date(),col("joining_date"))/12)).filter(col("exp")<=2).show()
employee_df.fillna({"manager_id":999}).show()
employee_df.filter(months_between(current_date(), col("joining_date")) <= 24).show()
employee_df.withColumn("salary_grade",when(col("salary")>=8000,"A").when(col("salary")>=60000,"B").otherwise("C")).show()
employee_df.withColumn("NEW_EMP_ID",concat(lit("EMP"),col("emp_id"))).show()
employee_df.filter(col("name").like("%e")).show()
employee_df.orderBy("joining_date").show()
employee_df.filter(months_between(current_date(),col("joining_date"))<=36).show()


+----------+------+----+---+------+---------+------------+----------+-------+
|department|emp_id|name|age|salary|     city|joining_date|manager_id|avg_sal|
+----------+------+----+---+------+---------+------------+----------+-------+
|        IT|   102| Bob| 30| 70000|Bangalore|  2021-06-20|       202|65000.0|
|        HR|   105| Eva| 35| 80000|     NULL|  2019-05-25|       201|62500.0|
+----------+------+----+---+------+---------+------------+----------+-------+

+------+----+---+----------+------+---------+------------+----------+
|emp_id|name|age|department|salary|     city|joining_date|manager_id|
+------+----+---+----------+------+---------+------------+----------+
|   102| Bob| 30|        IT| 70000|Bangalore|  2021-06-20|       202|
+------+----+---+----------+------+---------+------------+----------+

+---------+-----+
|     city|count|
+---------+-----+
|  Chennai|    2|
|Bangalore|    2|
+---------+-----+

+------+-----+---+----------+------+---------+------------+----------+-

- Challenge Questions
- 91.	Rank employees by salary within each department.
- 92.	Find the top 2 highest-paid employees in every department.
- 93.	Find departments with average salary greater than 60000.
- 94.	Find employees who do not have a manager.
- 95.	Display employees with manager names and department locations in a single DataFrame.
- 96.	Identify employees with missing information (NULL values).
- 97.	Replace missing salaries with the department average salary.
- 98.	Calculate monthly tax as 10% of salary.
- 99.	Find employees who joined on weekends.
- 100.	Create a final report containing:
- •	Employee Name
- •	Department
- •	Manager Name
- •	Department Location
- •	Salary
- •	Bonus
- •	Annual Salary
- •	Years Worked
- •	Salary Grade


In [0]:
win = Window.orderBy(col("salary").desc())
employee_df.withColumn("Rank",rank().over(win)).show()
employee_df.withColumn("Rank",rank().over(win)).limit(2).show()
employee_df.groupBy("department").avg("salary").filter(col("avg(salary)")>60000).show()
employee_df.filter(col("manager_id").isNull()).show()
employee_df.join(department_df,employee_df.department == department_df.dept_name,"left").join(manager_df,employee_df.manager_id == manager_df.manager_id).select("name","manager_name","location").show()
employee_df.filter(col("name").isNull() |
                   col("department").isNull() |
                   col("manager_id").isNull() |
                   col("salary").isNull() |
                   col("joining_date").isNull() |
                   col("emp_id").isNull() |
                   col("age").isNull() |
                   col("city").isNull()).show()
dept_avg = employee_df.groupBy("department").agg(avg("salary").alias("avg_salary"))
employee_df.withColumn("monthly_tax",(col("salary")*0.1)).show()
employee_df.filter(dayofweek(col("joining_date")).isin(1,7)).show()
employee_df = employee_df.join(dept_avg, "department", "left").withColumn("salary",coalesce(col("salary"), col("avg_salary"))).drop("avg_salary").show()



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+-------+----+----------+------+---------+------------+----------+----+
|emp_id|   name| age|department|salary|     city|joining_date|manager_id|Rank|
+------+-------+----+----------+------+---------+------------+----------+----+
|   105|    Eva|  35|        HR| 80000|     NULL|  2019-05-25|       201|   1|
|   102|    Bob|  30|        IT| 70000|Bangalore|  2021-06-20|       202|   2|
|   104|  David|  28|   Finance| 65000|   Mumbai|  2020-09-18|       203|   3|
|   108|  Henry|  26|        IT| 60000|Bangalore|  2024-01-10|      NULL|   4|
|   106|  Frank|  29| Marketing| 55000|Hyderabad|        NULL|       204|   5|
|   101|  Alice|  25|        HR| 45000|  Chennai|  2022-01-15|       201|   6|
|   103|Charlie|NULL|        IT|  NULL|  Chennai|  2023-03-12|       202|   7|
|   107|  Grace|  31|   Finance|  NULL|     Pune|  2022-12-01|       203|   7|
+------+-------+----+----------+------+---------+------------+----------+----+

+------+----+---+----------+------+---------+------

In [0]:
final_report = employee_df.join(department_df,employee_df.department == department_df.dept_name,"left"
    ) \
    .join(manager_df,employee_df.manager_id == manager_df.manager_id,"left"
    ) \
    .withColumn("Bonus",col("salary") * 0.20
    ) \
    .withColumn("Annual Salary",col("salary") * 12
    ) \
    .withColumn("Years Worked",floor(months_between(current_date(), col("joining_date")) / 12)
    ) \
    .withColumn("Salary Grade",when(col("salary") >= 80000, "A").when(col("salary") >= 60000, "B").otherwise("C")
    ) \
    .select(
        col("name").alias("Employee Name"),
        col("department").alias("Department"),
        col("manager_name").alias("Manager Name"),
        col("location").alias("Department Location"),
        col("salary").alias("Salary"),
        col("Bonus"),
        col("Annual Salary"),
        col("Years Worked"),
        col("Salary Grade")
    ).show()

+-------------+----------+------------+-------------------+------+-------+-------------+------------+------------+
|Employee Name|Department|Manager Name|Department Location|Salary|  Bonus|Annual Salary|Years Worked|Salary Grade|
+-------------+----------+------------+-------------------+------+-------+-------------+------------+------------+
|        Alice|        HR|      Robert|            Chennai| 45000| 9000.0|       540000|           4|           C|
|          Bob|        IT|    Jennifer|          Bangalore| 70000|14000.0|       840000|           5|           B|
|      Charlie|        IT|    Jennifer|          Bangalore|  NULL|   NULL|         NULL|           3|           C|
|        David|   Finance|     Michael|             Mumbai| 65000|13000.0|       780000|           5|           B|
|          Eva|        HR|      Robert|            Chennai| 80000|16000.0|       960000|           7|           A|
|        Frank| Marketing|      Sophia|          Hyderabad| 55000|11000.0|      